#  B1 — Everyday Object Classifier

**Classes:** Pen, Bottle, Phone, Book, Key

This notebook loads the exported Teachable Machine image model, performs the required 224×224 preprocessing and [-1, 1] normalization, and deploys it with an OpenCV webcam.

In [13]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

print("OpenCV Version:", cv2.__version__)
print("NumPy Version:", np.__version__)
print("TensorFlow Version:", tf.__version__)

OpenCV Version: 4.9.0
NumPy Version: 1.26.4
TensorFlow Version: 2.15.1


##  Model Paths



In [14]:
PROJECT_DIR = Path(r"C:\Users\patel\OneDrive\Desktop\R&W\Deep Learning\DL_PR4B")
MODEL_PATH = PROJECT_DIR / "models" / "image" / "keras_model.h5"
LABELS_PATH = PROJECT_DIR / "models" / "image" / "labels.txt"

print("Model exists:", MODEL_PATH.exists())
print("Labels exist:", LABELS_PATH.exists())

Model exists: True
Labels exist: True


In [15]:
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    class_names = [line.strip() for line in f.readlines()]

print("Classes:")
for i, name in enumerate(class_names):
    print(f"{i}: {name}")

Classes:
0: 0 pen
1: 1 bottle
2: 2 key
3: 3 book
4: 4 phone


In [16]:
model = tf.keras.models.load_model(str(MODEL_PATH), compile=False)
print("Model loaded successfully! ✅")
model.summary()

Model loaded successfully! ✅
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_1 (Sequential)   (None, 1280)              410208    
                                                                 
 sequential_3 (Sequential)   (None, 5)                 128600    
                                                                 
Total params: 538808 (2.06 MB)
Trainable params: 524728 (2.00 MB)
Non-trainable params: 14080 (55.00 KB)
_________________________________________________________________


##  Prediction Function



In [17]:
def predict_image(image_bgr):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    image_resized = cv2.resize(image_rgb, (224, 224))
    image_array = np.asarray(image_resized, dtype=np.float32)
    normalized = (image_array / 127.5) - 1.0
    input_data = np.expand_dims(normalized, axis=0)

    prediction = model.predict(input_data, verbose=0)[0]
    index = int(np.argmax(prediction))
    confidence = float(prediction[index])
    label = class_names[index] if index < len(class_names) else f"Class {index}"
    return label, confidence, prediction

##  Webcam Deployment


In [18]:
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam. Close other apps using the camera and try again.")

print("Webcam started. Show Pen, Bottle, Phone, Book, or Key.")
print("Press Q to quit.")
last_frame = None

while True:
    ret, frame = cap.read()
    if not ret:
        print("Could not read webcam frame.")
        break

    label, confidence, _ = predict_image(frame)
    text = f"{label}: {confidence * 100:.1f}%"

    cv2.rectangle(frame, (10, 10), (540, 75), (0, 0, 0), -1)
    cv2.putText(frame, text, (25, 55), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA)
    cv2.putText(frame, "Press Q to quit", (25, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.imshow("B1 Everyday Object Classifier", frame)
    last_frame = frame.copy()

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

if last_frame is not None:
    result_path = OUTPUT_DIR / "b1_image_result.jpg"
    cv2.imwrite(str(result_path), last_frame)
    print("Saved:", result_path)

Webcam started. Show Pen, Bottle, Phone, Book, or Key.
Press Q to quit.
Saved: C:\Users\patel\OneDrive\Desktop\R&W\Deep Learning\DL_PR4B\output\b1_image_result.jpg
